# Full PDF VLM Question Answering - Qwen2.5-VL

This notebook:
- Runs on Google Colab T4 GPU
- Uploads any PDF
- Converts the full PDF into page images
- Asks your question across the whole PDF
- Does **not** hardcode any page number
- Saves the output to `vlm_extracted_output.txt`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 1: Install required libraries
!pip install -q pymupdf pillow transformers accelerate sentencepiece qwen-vl-utils safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 15.3 MB/s eta 0:00:00


In [ ]:
# STEP 2: Imports
import os
import gc
import fitz
import torch
from PIL import Image
from google.colab import files
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration


In [ ]:
# STEP 3: Check GPU
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Please enable Colab T4 GPU: Runtime > Change runtime type > T4 GPU")


CUDA Available: True
GPU: Tesla T4


In [ ]:
# STEP 4: Upload PDF
# Upload your PDF here. Example: test.pdf

uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_files:
    raise FileNotFoundError("No PDF file uploaded. Please upload a .pdf file.")

PDF_PATH = pdf_files[0]

print("Using PDF:", PDF_PATH)


Saving test.pdf to test.pdf
Using PDF: test.pdf


In [ ]:
# STEP 5: Load Qwen Vision Language Model
# First run may take time because the model will be downloaded.

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(model_id)

print("Loading model...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully.")


Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# STEP 6: Convert FULL PDF to images
# This converts every page. No page number is hardcoded.

def pdf_to_images(pdf_path, zoom=1.25):
    doc = fitz.open(pdf_path)
    images = []

    print(f"Total pages: {len(doc)}")

    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))

        img = Image.frombytes(
            "RGB",
            (pix.width, pix.height),
            pix.samples
        )

        images.append(img)
        print(f"Converted page {page_num + 1}")

    doc.close()
    return images

images = pdf_to_images(PDF_PATH)

print(f"Total converted images: {len(images)}")


Total pages: 11
Converted page 1
Converted page 2
Converted page 3
Converted page 4
Converted page 5
Converted page 6
Converted page 7
Converted page 8
Converted page 9
Converted page 10
Converted page 11
Total converted images: 11


In [ ]:
# STEP 7: Ask question to every PDF page safely
# It scans the full PDF and keeps only relevant page answers.

def clean_model_response(response_text):
    # Remove repeated prompt-like content when model returns chat template text
    if "assistant" in response_text.lower():
        parts = response_text.split("assistant")
        response_text = parts[-1].strip()
    return response_text.strip()


def ask_model_full_pdf(images, question, max_new_tokens=160):
    relevant_answers = []

    for i, img in enumerate(images):
        print(f"Processing page {i + 1}/{len(images)}...")

        page_prompt = f"""You are reading one page from a PDF.

Answer the user's question only if this page contains the required information.

If this page does not contain the answer, reply exactly:
NOT_FOUND

User question:
{question}

Important:
- Be accurate with numbers, percentages, table values, and labels.
- Do not guess.
- If the page has a table, preserve the correct row-column relationship.
- If the answer is present, give a concise answer and mention the page number.
"""

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": page_prompt}
                ],
            }
        ]

        try:
            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = processor(
                text=[text],
                images=[img],
                return_tensors="pt"
            ).to(model.device)

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False
                )

            response = processor.decode(output_ids[0], skip_special_tokens=True)
            response = clean_model_response(response)

            if response and "NOT_FOUND" not in response.upper():
                relevant_answers.append(
                    f"Page {i + 1}:\n{response}"
                )

            del inputs, output_ids
            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(f"Error while processing page {i + 1}: {e}")
            torch.cuda.empty_cache()
            gc.collect()
            continue

    if not relevant_answers:
        return "Answer not found in the PDF."

    return "\n\n".join(relevant_answers)


In [ ]:
# STEP 8: Interactive Question Loop

while True:

    question = input("\nEnter your question (or type 'exit'): ")

    if question.lower() == "exit":
        print("Exiting question loop...")
        break

    answer = ask_model_full_pdf(images, question)

    print("\n================ FINAL ANSWER ================\n")
    print(answer)

    with open("vlm_extracted_output.txt", "a", encoding="utf-8") as f:
        f.write("\n\n====================================\n")
        f.write(f"QUESTION:\n{question}\n\n")
        f.write(f"ANSWER:\n{answer}\n")

    print("\nAnswer appended to vlm_extracted_output.txt")


Enter your question (or type 'exit'): How many countries sell Unilever products?
Processing page 1/11...
Processing page 2/11...
Processing page 3/11...
Processing page 4/11...
Processing page 5/11...
Processing page 6/11...
Processing page 7/11...
Processing page 8/11...
Processing page 9/11...
Processing page 10/11...
Processing page 11/11...

================ FINAL ANSWER ================

Page 2:
190

Page number: 2

Answer appended to vlm_extracted_output.txt

Enter your question (or type 'exit'): exit
Exiting question loop...


In [ ]:
# STEP 8: Ask your question
# You can ask any question. No page is hardcoded.

question = input("Enter your question: ")

answer = ask_model_full_pdf(images, question)

output_file = "vlm_extracted_output.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("QUESTION:\n")
    f.write(question)
    f.write("\n\nANSWER:\n")
    f.write(answer)

print("\n================ FINAL ANSWER ================\n")
print(answer)

print(f"\nSaved output to: {output_file}")


Enter your question: What are the 7 strategic growth priorities?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processing page 1/11...
Processing page 2/11...
Processing page 3/11...
Processing page 4/11...
Processing page 5/11...
Processing page 6/11...
Processing page 7/11...
Processing page 8/11...
Processing page 9/11...
Processing page 10/11...
Processing page 11/11...

================ FINAL ANSWER ================

Page 4:
The 7 strategic growth priorities are:

1. Beauty
2. Premium
3. United States
4. Wellbeing
5. Digital Commerce
6. India
7. Personal Care

Page number: 5

Saved output to: vlm_extracted_output.txt


In [ ]:
# STEP 9: Download output file
#files.download("vlm_extracted_output.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>